# From Basics to Insights: Mastering Pandas for Economic Analysis

**A practical project applying Pandas (basics → intermediate → advanced) to a classic Economics problem:**  
**Understanding Inflation, Real Wages, Cost-of-Living Pressures, and Inequality.**

Inspired by the structure of [Jake VanderPlas' Python Data Science Handbook – Chapter 3 (Pandas)](https://jakevdp.github.io/PythonDataScienceHandbook/03.00-introduction-to-pandas.html).

### The Common Economics Issue We Solve
Households and policymakers constantly ask:

- How fast is the cost of living rising?
- Are nominal wage increases keeping up with inflation? (i.e., what is happening to **real wages**?)
- Which categories (food, housing, energy, transport) are driving inflation?
- Do different income groups or regions experience the same cost-of-living pressure?
- Is inequality widening or narrowing?

We build a complete analytical pipeline that answers these questions using Pandas.

### Learning Path
| Level | Pandas Topics Covered | Economics Application |
|-------|-----------------------|-----------------------|
| **Basics** | Series, DataFrame, Indexing & Selection | Economic indicator tables, selecting high-inflation periods |
| **Intermediate** | Operations, Missing Data, Hierarchical Indexing, Merge/Join | Real wage calculation, combining CPI + wages + unemployment |
| **Advanced** | GroupBy, Pivot Tables, Strings, Time Series, `query`/`eval` | Category contributions, inequality metrics, trend detection |

> **Emphasis throughout**: Every technique is used to extract **economic insights**, not just to demonstrate syntax.


## 1. Environment Setup


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', 16)
pd.set_option('display.width', 120)
pd.set_option('display.float_format', '{:.2f}'.format)

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12

print(f"Pandas version : {pd.__version__}")
print(f"NumPy version  : {np.__version__}")
print("Environment ready for economic analysis.")


## 2. Creating Pandas Objects – Core Economic Indicators

We start with the fundamental building blocks of macroeconomic analysis:  
price indices, wage series, and labor market indicators.


In [ ]:
# ------------------------------------------------------------------
# Synthetic but realistic national economic indicators (monthly, 2019-2025)
# ------------------------------------------------------------------
rng = np.random.default_rng(42)
dates = pd.date_range('2019-01-01', '2025-12-01', freq='MS')
n = len(dates)

# Base CPI (overall) with realistic inflation path: low → COVID dip → high inflation → moderation
t = np.arange(n)
base_cpi = 100 * np.cumprod(1 + np.concatenate([
    np.full(12, 0.0015),      # 2019 mild
    np.full(12, 0.0010),      # 2020
    np.full(12, 0.0045),      # 2021 rising
    np.full(12, 0.0065),      # 2022 peak inflation
    np.full(12, 0.0035),      # 2023
    np.full(12, 0.0025),      # 2024
    np.full(12, 0.0020),      # 2025
]))

# Category CPIs (different dynamics)
food_cpi     = base_cpi * (1 + 0.08 * np.sin(t/8) + rng.normal(0, 0.01, n).cumsum()*0.3)
housing_cpi  = base_cpi * (1 + 0.04 * np.sin(t/10) + 0.015 * t/n)
energy_cpi   = base_cpi * (1 + 0.25 * np.sin(t/6) + rng.normal(0, 0.03, n).cumsum()*0.4)
transport_cpi= base_cpi * (1 + 0.12 * np.sin(t/7) + rng.normal(0, 0.015, n).cumsum()*0.25)
core_cpi     = base_cpi * (1 + 0.02 * np.sin(t/12))   # excluding food & energy

cpi = pd.DataFrame({
    'overall': base_cpi,
    'food': food_cpi,
    'housing': housing_cpi,
    'energy': energy_cpi,
    'transport': transport_cpi,
    'core': core_cpi
}, index=dates)

# Nominal average wages (monthly)
wage_growth = np.concatenate([
    np.full(24, 0.0025),   # 2019-2020
    np.full(12, 0.0040),   # 2021
    np.full(12, 0.0055),   # 2022 catch-up
    np.full(24, 0.0035),   # 2023-2024
    np.full(12, 0.0030),   # 2025
])
nominal_wage = 3200 * np.cumprod(1 + wage_growth + rng.normal(0, 0.001, n))

# Unemployment rate
unemp = 5.8 + 3.5 * np.exp(-((t-20)/8)**2) + rng.normal(0, 0.15, n)  # COVID spike
unemp = np.clip(unemp, 3.5, 12)

econ = pd.DataFrame({
    'cpi_overall': cpi['overall'],
    'nominal_wage': nominal_wage,
    'unemployment': unemp
}, index=dates)

print("Core economic indicators (sample):")
display(econ.head(8).round(2))
print("\nCPI by category (sample):")
display(cpi.head(6).round(2))


## 3. Data Indexing and Selection

Selecting specific periods (e.g., high-inflation years) and categories is fundamental in economic reporting.


In [ ]:
# 3.1 High-inflation period (2021-2023)
high_inf = econ.loc['2021':'2023']
print("Economic indicators during the high-inflation period (2021-2023):")
display(high_inf.describe().round(2))

# 3.2 Latest available data
print("\nMost recent observation:")
display(econ.iloc[[-1]].round(2))

# 3.3 Select specific CPI categories
print("\nFood and Energy CPI (key volatile components):")
display(cpi[['food', 'energy']].loc['2022'].head(6).round(2))

# 3.4 Boolean selection – months with unemployment > 7%
high_unemp = econ[econ['unemployment'] > 7]
print(f"\nMonths with unemployment > 7%: {len(high_unemp)}")
print(f"Average CPI overall in those months: {high_unemp['cpi_overall'].mean():.1f}")

# Visual insight
fig, ax1 = plt.subplots(figsize=(12, 5))
ax1.plot(econ.index, econ['cpi_overall'], color='steelblue', label='CPI Overall')
ax1.set_ylabel('CPI (index)', color='steelblue')
ax2 = ax1.twinx()
ax2.plot(econ.index, econ['unemployment'], color='crimson', alpha=0.7, label='Unemployment %')
ax2.set_ylabel('Unemployment rate (%)', color='crimson')
ax1.set_title('CPI vs Unemployment Rate')
fig.legend(loc='upper left', bbox_to_anchor=(0.1, 0.9))
plt.tight_layout()
plt.show()


## 4. Operating on Data in Pandas – Real Wages & Inflation

The most important derived economic metric for households is the **real wage**  
(nominal wage deflated by the price level).


In [ ]:
# 4.1 Year-over-year inflation rates
cpi_yoy = cpi.pct_change(12) * 100   # YoY %
print("Year-over-year inflation by category (%):")
display(cpi_yoy.dropna().tail(8).round(2))

# 4.2 Real wage (base = first observation)
econ['real_wage'] = econ['nominal_wage'] / (econ['cpi_overall'] / econ['cpi_overall'].iloc[0])
econ['real_wage_index'] = econ['real_wage'] / econ['real_wage'].iloc[0] * 100

# 4.3 Real wage growth (YoY)
econ['real_wage_yoy'] = econ['real_wage'].pct_change(12) * 100

print("\nNominal vs Real wage (sample):")
display(econ[['nominal_wage', 'real_wage', 'real_wage_index', 'real_wage_yoy']].dropna().tail(8).round(2))

# Visual – the key economic insight
fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

econ['nominal_wage'].plot(ax=axes[0], label='Nominal wage', color='steelblue')
econ['real_wage'].plot(ax=axes[0], label='Real wage (CPI-deflated)', color='darkorange')
axes[0].set_ylabel('Wage units')
axes[0].set_title('Nominal vs Real Wages')
axes[0].legend()

cpi_yoy['overall'].plot(ax=axes[1], color='crimson', label='Overall inflation (YoY %)')
econ['real_wage_yoy'].plot(ax=axes[1], color='green', label='Real wage growth (YoY %)')
axes[1].axhline(0, color='gray', linestyle='--', linewidth=0.8)
axes[1].set_ylabel('%')
axes[1].set_title('Inflation vs Real Wage Growth')
axes[1].legend()
plt.tight_layout()
plt.show()

print("\n--- Key Insight ---")
print("When inflation exceeds nominal wage growth, real wages fall and purchasing power declines.")
print("This is one of the most closely watched relationships in applied macroeconomics.")


## 5. Handling Missing Data

Economic survey and administrative data frequently contain gaps  
(late reporting, revised series, temporary suspension of collection).


In [ ]:
# Simulate a wage series with realistic missing observations
wage_series = econ['nominal_wage'].copy()
missing_idx = rng.choice(wage_series.index, size=9, replace=False)
wage_series.loc[missing_idx] = np.nan

print("Wage series with missing values:")
print(f"Missing observations: {wage_series.isna().sum()}")
display(wage_series.loc['2022-06':'2023-03'])

# Common strategies in economic time series
print("\n1. Linear interpolation (often acceptable for monthly wages):")
filled = wage_series.interpolate(method='time')
print(f"   Missing after interpolation: {filled.isna().sum()}")

print("\n2. Forward-fill (used when the last known value is still valid):")
filled_ffill = wage_series.ffill(limit=2)

# Visual
fig, ax = plt.subplots(figsize=(12, 4))
wage_series.plot(ax=ax, label='Original (with gaps)', alpha=0.6, color='gray', marker='o', markersize=3)
filled.plot(ax=ax, label='Time-interpolated', color='teal', linewidth=1.5)
ax.set_title('Nominal Wage Series – Handling Missing Observations')
ax.set_ylabel('Wage')
ax.legend()
plt.tight_layout()
plt.show()

print("\n--- Insight ---")
print("In official statistics, missing values are carefully flagged and imputation methods")
print("are documented. For analysis we usually interpolate short gaps and leave longer ones missing.")


## 6. Hierarchical Indexing (MultiIndex)

Economic data is naturally multi-dimensional:  
**Region × Income Quintile**, **Category × Year**, **Sector × Indicator**.


In [ ]:
# Create a MultiIndex dataset: Average inflation by Region and Category (illustrative)
regions = ['North', 'Central', 'South', 'Coast']
categories = ['Food', 'Housing', 'Energy', 'Transport', 'Core']

# Synthetic regional category inflation (annual average for a recent year)
data = []
for reg in regions:
    for cat in categories:
        # Different regional pressures
        base = {'Food': 6.8, 'Housing': 5.2, 'Energy': 11.5, 'Transport': 7.1, 'Core': 4.3}[cat]
        regional_shock = {'North': 0.8, 'Central': 0.0, 'South': 1.5, 'Coast': -0.4}[reg]
        val = base + regional_shock + rng.normal(0, 0.4)
        data.append({'region': reg, 'category': cat, 'inflation_yoy': val})

regional_inf = pd.DataFrame(data).set_index(['region', 'category'])['inflation_yoy']
print("MultiIndex Series – YoY Inflation by Region × Category:")
display(regional_inf.round(2))

print("\nAll categories in the South region (highest pressure):")
display(regional_inf.loc['South'].round(2))

print("\nCross-section – Food inflation across regions:")
display(regional_inf.xs('Food', level='category').round(2))

# Unstack for matrix view
inf_matrix = regional_inf.unstack()
print("\nInflation matrix (Region × Category):")
display(inf_matrix.round(2))

# Heatmap
fig, ax = plt.subplots(figsize=(9, 5))
sns.heatmap(inf_matrix, annot=True, fmt='.1f', cmap='YlOrRd', ax=ax)
ax.set_title('YoY Inflation (%) by Region and Category')
plt.tight_layout()
plt.show()

print("\n--- Insight ---")
print("The South region faces systematically higher inflation, especially in Food and Energy.")
print("This has direct implications for regional poverty rates and targeted social transfers.")


## 7. Combining Datasets – Merge & Join

In practice we combine multiple official sources:
- CPI (national statistics office)
- Wage surveys / administrative tax data
- Labor force survey (unemployment)
- Sometimes household expenditure surveys


In [ ]:
# 7.1 Create a simple household income-group dataset (annual)
years = list(range(2019, 2026))
income_groups = ['Q1 (lowest)', 'Q2', 'Q3', 'Q4', 'Q5 (highest)']

# Synthetic real income indices by quintile (base 2019 = 100)
# Lower quintiles suffered more during high inflation
records = []
for y in years:
    for i, g in enumerate(income_groups):
        # Differential real income growth
        growth = [0.8, 1.2, 1.6, 2.1, 2.8][i]   # higher quintiles recover faster
        if y >= 2022:
            growth -= [1.8, 1.2, 0.7, 0.3, 0.0][i]  # inflation hit lower groups harder
        base = 100 * (1 + growth/100)**(y-2019)
        records.append({'year': y, 'income_group': g, 'real_income_index': base + rng.normal(0, 0.8)})

hh = pd.DataFrame(records)

print("Household real income index by quintile (sample):")
display(hh.head(10).round(2))

# 7.2 Annual aggregates from monthly econ data
annual = econ.resample('YE').mean()
annual.index = annual.index.year
annual = annual.rename_axis('year').reset_index()

# 7.3 Merge
merged = pd.merge(hh, annual[['year', 'cpi_overall', 'unemployment', 'real_wage_index']],
                  on='year', how='left')

print("\nMerged household + macro indicators:")
display(merged.head(10).round(2))


## 8. Aggregation and Grouping

Core economic questions:
- What was average inflation by year?
- How did real income evolve for each quintile?
- Which categories contributed most to overall inflation?


In [ ]:
# 8.1 Annual average inflation
annual_inf = cpi_yoy.resample('YE').mean()
annual_inf.index = annual_inf.index.year
print("Average YoY inflation by year and category (%):")
display(annual_inf.round(2))

# 8.2 Real income by income group over time
print("\nAverage real income index by quintile:")
quintile_avg = hh.groupby('income_group')['real_income_index'].mean()
display(quintile_avg.round(2))

# Evolution
pivot_hh = hh.pivot(index='year', columns='income_group', values='real_income_index')
print("\nReal income index evolution:")
display(pivot_hh.round(1))

# Visual
pivot_hh.plot(figsize=(11, 5), marker='o')
plt.title('Real Income Index by Income Quintile (2019 = ~100)')
plt.ylabel('Index')
plt.axhline(100, color='gray', linestyle='--', linewidth=0.8)
plt.legend(title='Income group', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

print("\n--- Key Insight ---")
print("Lower-income quintiles (Q1–Q2) experienced a clearer decline in real purchasing power")
print("during the high-inflation years, while higher quintiles recovered faster.")
print("This is a classic channel through which inflation increases inequality.")


## 9. Pivot Tables

Pivot tables are the workhorse of economic reporting  
(e.g., “Inflation by Category × Year”, “Real income by Quintile × Year”).


In [ ]:
# Classic pivot already shown above; here we add margins and more views
pivot_inf = cpi_yoy.copy()
pivot_inf['year'] = pivot_inf.index.year
pivot_inf = pivot_inf.groupby('year').mean()

print("Inflation pivot (Year × Category) with overall:")
display(pivot_inf.round(2))

# Heatmap of inflation
fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(pivot_inf.T, annot=True, fmt='.1f', cmap='RdYlGn_r', center=0, ax=ax)
ax.set_title('YoY Inflation (%) Heatmap – Category × Year')
plt.tight_layout()
plt.show()

# Contribution-style view (simplified): which categories ran hottest
print("\nHottest inflation categories by year (rank):")
for year in pivot_inf.index:
    top = pivot_inf.loc[year].drop('overall', errors='ignore').nlargest(2)
    print(f"  {year}: {top.index[0]} ({top.iloc[0]:.1f}%), {top.index[1]} ({top.iloc[1]:.1f}%)")


## 10. Working with Strings

Economic series names, regional codes, and publication labels often need cleaning.


In [ ]:
# Example: clean and standardize indicator names
indicators = pd.Series([
    '  CPI - Overall  ', 'cpi_food', 'Housing_CPI', 'ENERGY',
    'Transport Prices', 'Core Inflation (ex food & energy)',
    'Nominal Average Wage', 'Unemployment Rate (%)'
], name='raw_name')

print("Raw indicator names:")
print(indicators.tolist())

# Cleaning pipeline
clean = (indicators
         .str.strip()
         .str.lower()
         .str.replace(r'[^a-z0-9]+', '_', regex=True)
         .str.strip('_'))

print("\nCleaned names:")
print(clean.tolist())

# Extract category from longer labels
labels = pd.Series([
    'North-Food-2023', 'Central-Housing-2024', 'South-Energy-2022',
    'Coast-Transport-2025', 'North-Core-2023'
])
print("\nParsed region and category:")
parsed = labels.str.split('-', expand=True)
parsed.columns = ['region', 'category', 'year']
display(parsed)


## 11. Working with Time Series

Almost all macroeconomic analysis is time-series analysis.


In [ ]:
# 11.1 Resampling & frequency conversion
print("Quarterly average inflation (overall):")
quarterly_inf = cpi_yoy['overall'].resample('QE').mean()
display(quarterly_inf.dropna().tail(8).round(2))

# 11.2 Rolling windows – smoothed trends
econ['cpi_yoy_smooth'] = cpi_yoy['overall'].rolling(6, center=True).mean()
econ['real_wage_yoy_smooth'] = econ['real_wage_yoy'].rolling(6, center=True).mean()

fig, ax = plt.subplots(figsize=(12, 5))
cpi_yoy['overall'].plot(ax=ax, alpha=0.4, label='YoY Inflation (raw)', color='crimson')
econ['cpi_yoy_smooth'].plot(ax=ax, label='6-month rolling mean', color='darkred', linewidth=2)
econ['real_wage_yoy_smooth'].plot(ax=ax, label='Real wage growth (smoothed)', color='green', linewidth=2)
ax.axhline(0, color='gray', linestyle='--', linewidth=0.8)
ax.set_ylabel('%')
ax.set_title('Inflation and Real Wage Growth – Smoothed Trends')
ax.legend()
plt.tight_layout()
plt.show()

# 11.3 Lag features (useful for simple forecasting or Phillips-curve style analysis)
econ['unemp_lag3'] = econ['unemployment'].shift(3)
econ['inf_lag6'] = cpi_yoy['overall'].shift(6)

print("\nSample with lags (useful for regression later):")
display(econ[['unemployment', 'unemp_lag3', 'cpi_overall']].dropna().tail(6).round(2))

# 11.4 Largest inflation accelerations
print("\nLargest month-over-month increases in overall CPI:")
print((cpi['overall'].pct_change() * 100).nlargest(5).round(2))


## 12. High-Performance Operations: `query` and `eval`

When working with large micro-data (household surveys with hundreds of thousands of rows),  
`query` and `eval` improve both speed and readability.


In [ ]:
# Simulate a larger household-level dataset
n_hh = 50_000
rng = np.random.default_rng(123)

hh_large = pd.DataFrame({
    'income': rng.lognormal(10.5, 0.7, n_hh),
    'region': rng.choice(['North', 'Central', 'South', 'Coast'], n_hh),
    'hh_size': rng.integers(1, 7, n_hh),
    'inflation_exposure': rng.normal(5.5, 2.0, n_hh).clip(0),  # personal inflation rate
    'has_wage_increase': rng.binomial(1, 0.45, n_hh)
})

print(f"Large household DataFrame shape: {hh_large.shape}")

# Classic boolean indexing
%timeit -n 5 -r 2 hh_large[(hh_large['income'] < 25000) & (hh_large['inflation_exposure'] > 7) & (hh_large['region'] == 'South')]

# query
%timeit -n 5 -r 2 hh_large.query('income < 25000 and inflation_exposure > 7 and region == "South"')

# eval for derived metrics
hh_large['real_income_proxy'] = hh_large.eval('income / (1 + inflation_exposure/100)')
hh_large['vulnerable'] = hh_large.eval('(income < 30000) & (inflation_exposure > 6)')

print("\nVulnerable households (low income + high personal inflation):")
print(f"  Count: {hh_large['vulnerable'].sum():,}")
print(f"  Share: {hh_large['vulnerable'].mean()*100:.1f}%")

print("\n--- Insight ---")
print("Identifying households that face both low income and high inflation exposure")
print("is a classic targeting problem for social policy and cash-transfer design.")


## 13. Synthesis – Answering the Core Economic Questions

We now bring the pieces together to answer the questions posed at the beginning.


In [ ]:
# 13.1 Summary statistics for the high-inflation window
window = econ.loc['2021-01':'2023-12']
print("=== HIGH-INFLATION WINDOW (2021–2023) ===")
print(f"Average YoY inflation (overall) : {cpi_yoy.loc['2021':'2023', 'overall'].mean():.1f}%")
print(f"Peak YoY inflation              : {cpi_yoy.loc['2021':'2023', 'overall'].max():.1f}%")
print(f"Average real wage growth (YoY)  : {window['real_wage_yoy'].mean():.1f}%")
print(f"Change in real wage index       : {window['real_wage_index'].iloc[-1] - window['real_wage_index'].iloc[0]:.1f} points")

# 13.2 Category contribution (simplified view)
print("\n=== WHICH CATEGORIES DROVE INFLATION? ===")
cat_avg = cpi_yoy.loc['2021':'2023', ['food', 'housing', 'energy', 'transport', 'core']].mean().sort_values(ascending=False)
display(cat_avg.round(2))

# 13.3 Inequality signal
print("\n=== INEQUALITY SIGNAL (Real Income by Quintile) ===")
print("Real income index change 2019 → 2025:")
change = pivot_hh.loc[2025] - pivot_hh.loc[2019]
display(change.round(1))

# 13.4 Final visual dashboard
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

# Inflation path
cpi_yoy['overall'].plot(ax=axes[0,0], color='crimson')
axes[0,0].axhline(0, color='gray', linestyle='--', lw=0.8)
axes[0,0].set_title('Overall YoY Inflation')
axes[0,0].set_ylabel('%')

# Real wage index
econ['real_wage_index'].plot(ax=axes[0,1], color='darkorange')
axes[0,1].axhline(100, color='gray', linestyle='--', lw=0.8)
axes[0,1].set_title('Real Wage Index (start = 100)')

# Category inflation during peak
cat_avg.plot(kind='barh', ax=axes[1,0], color='steelblue')
axes[1,0].set_title('Avg YoY Inflation by Category (2021-23)')
axes[1,0].set_xlabel('%')

# Quintile outcomes
change.plot(kind='bar', ax=axes[1,1], color='seagreen')
axes[1,1].axhline(0, color='gray', linestyle='--', lw=0.8)
axes[1,1].set_title('Real Income Index Change 2019→2025 by Quintile')
axes[1,1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

print("\n=== EXECUTIVE CONCLUSIONS ===")
print(
"1. Inflation peaked in 2022; energy and food were the main drivers.\n"
"2. Real wages stagnated or fell during the high-inflation window —\n"
"   nominal wage growth did not fully compensate for price increases.\n"
"3. Lower-income households experienced a larger erosion of purchasing power.\n"
"4. This combination (high inflation + uneven real-income effects) is a classic\n"
"   mechanism that temporarily widens inequality and raises the political salience\n"
"   of cost-of-living measures."
)


## 14. Mapping Back to the Pandas Handbook

| Handbook Section | Technique Demonstrated | Economics Insight Gained |
|------------------|------------------------|--------------------------|
| 03.01 Introducing Pandas Objects | Series & DataFrame creation | CPI, wages, unemployment as core objects |
| 03.02 Data Indexing & Selection | `.loc`, boolean, fancy | Selecting high-inflation windows & categories |
| 03.03 Operations | Vectorized arithmetic, pct_change | Real wages, YoY inflation rates |
| 03.04 Missing Values | `isna`, `interpolate`, `ffill` | Realistic survey / revision gaps |
| 03.05 Hierarchical Indexing | MultiIndex, `xs`, `unstack` | Region × Category inflation matrix |
| 03.06 Concat & Append | (construction of series) | Building multi-year indicator panels |
| 03.07 Merge & Join | `merge`, `join` | Combining macro indicators with household data |
| 03.08 Aggregation & Grouping | `groupby`, `resample` | Annual averages, quintile outcomes |
| 03.09 Pivot Tables | `pivot`, `pivot_table` | Year × Category and Year × Quintile reports |
| 03.10 Working with Strings | `.str` accessor | Cleaning indicator and regional labels |
| 03.11 Time Series | resample, rolling, shift, pct_change | Trends, lags, acceleration detection |
| 03.12 Performance | `query`, `eval` | Fast filtering of vulnerable households |

---

### Next Steps for Real Economic Analysis
1. Replace synthetic series with official data (national statistics office, central bank, OECD, World Bank, ILO).
2. Add more granular household survey micro-data for distributional analysis.
3. Estimate simple Phillips-curve or wage-price pass-through regressions.
4. Construct alternative price indices (e.g., democratic CPI that weights lower-income baskets more heavily).
5. Move from descriptive analysis to causal or forecasting models.

**You now have a complete template that follows the Pandas handbook progression while tackling one of the most common and policy-relevant questions in applied economics.**


## 15. Further Resources

- Jake VanderPlas – [Python Data Science Handbook, Chapter 3](https://jakevdp.github.io/PythonDataScienceHandbook/03.00-introduction-to-pandas.html)
- Official data sources:
  - National statistics offices (CPI, labor force surveys)
  - Central banks (inflation reports, wage indicators)
  - OECD, World Bank, IMF, ILO databases
  - Eurostat / BEA / BLS depending on country
- Classic references: “The Economics of Inflation”, distributional national accounts literature, cost-of-living indices research

---

*Notebook generated for practical Pandas mastery applied to a core Economics problem.*  
*All data is synthetic but designed to reproduce realistic macroeconomic patterns for teaching purposes.*
